# 05: Norms

RMSNorm and LayerNorm in int.

The hard part: $1/\sqrt{v}$ without float, which is done with C via `clz` + 256-entry LUT.

In [ ]:
import os
import sys

if "COLAB_GPU" in os.environ:
    os.system("pip install -q git+https://github.com/PritRaj1/smelt.git")
    os.system("pip install -q 'transformers>=4.51,<5'")
else:
    sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd())))

import torch

from smelt.norm import layernorm_int32, rmsnorm_int32
from smelt.plac import from_fixed, to_fixed

## Int inverse square root

Decompose $v$ into mantissa $m \in [1, 2)$ and exponent $k$:
$$v = m \cdot 2^k$$

Then:
$$1/\sqrt{v} = \text{LUT}[m] \cdot 2^{-k/2}$$

- `__builtin_clzll(v)` (count leading zeros): counts how many zero bits before the first 1. Gives $k$ = `63 - clz`. One x86 instruction (`lzcnt`), one cycle.
- LUT has 256 entries for $1/\sqrt{m}$ where $m \in [1, 2)$. 1KB, fits in L1.
- $2^{-k/2}$ is a right-shift for even $k$. For odd $k$, multiply by $1/\sqrt{2} \approx 46341/65536$.

## RMSNorm

$$y_i = \frac{x_i}{\text{RMS}(x)} \cdot \gamma_i \quad\text{where}\quad \text{RMS}(x) = \sqrt{\frac{1}{n}\sum x_i^2}$$

Integer steps:
1. Sum of squares (int64 accumulation)
2. `isqrt_fixed(mean_sq)` via clz + LUT
3. Scale each element: `x[i] * inv_rms * gamma[i] >> FRAC`

In [2]:
def fix(x):
    return torch.from_numpy(to_fixed(x.numpy()))


torch.manual_seed(0)
x = torch.randn(1, 64, dtype=torch.float64)
gamma = torch.ones(64, dtype=torch.float64)

rms = (x**2).mean(dim=-1, keepdim=True).sqrt()
y_ref = (x / rms * gamma).numpy().flatten()
y_int = from_fixed(rmsnorm_int32(fix(x), fix(gamma)).numpy()).flatten()

print(f"max error: {abs(y_ref - y_int).max():.4f}")
print(f"\n{'idx':>4s}  {'float':>8s}  {'int':>8s}  {'err':>8s}")
for i in range(8):
    print(f"{i:>4d}  {y_ref[i]:>8.4f}  {y_int[i]:>8.4f}  {abs(y_ref[i] - y_int[i]):>8.4f}")

max error: 0.0007

 idx     float       int       err
   0   -2.0628   -2.0633    0.0005
   1   -0.3332   -0.3333    0.0001
   2   -0.9471   -0.9474    0.0002
   3    0.8924    0.8926    0.0002
   4   -0.7893   -0.7895    0.0002
   5   -1.1388   -1.1391    0.0003
   6   -0.5564   -0.5566    0.0001
   7   -0.7736   -0.7738    0.0002


## LayerNorm

Same as RMSNorm but subtracts the mean first:
$$y_i = \frac{x_i - \mu}{\sigma} \cdot \gamma_i + \beta_i$$

One extra reduction (mean) and one extra subtraction per element.

In [3]:
beta = torch.zeros(64, dtype=torch.float64)
y_ln_ref = torch.nn.functional.layer_norm(x, [64], gamma, beta).numpy().flatten()
y_ln_int = from_fixed(layernorm_int32(fix(x), fix(gamma), fix(beta)).numpy()).flatten()

print(f"max error: {abs(y_ln_ref - y_ln_int).max():.4f}")

max error: 0.0042


## Accuracy across dims

In [4]:
torch.manual_seed(42)
gamma_1 = torch.ones(4096, dtype=torch.float64)
beta_0 = torch.zeros(4096, dtype=torch.float64)

print(f"{'dim':>6s}  {'RMS err':>8s}  {'LN err':>8s}")
for dim in [64, 256, 1024, 4096]:
    x = torch.randn(32, dim, dtype=torch.float64)
    g, b = gamma_1[:dim], beta_0[:dim]

    rms_ref = (x / (x.pow(2).mean(-1, keepdim=True).sqrt().clamp(min=1e-6)) * g).numpy()
    rms_int = from_fixed(rmsnorm_int32(fix(x), fix(g)).numpy())

    ln_ref = torch.nn.functional.layer_norm(x, [dim], g, b).numpy()
    ln_int = from_fixed(layernorm_int32(fix(x), fix(g), fix(b)).numpy())

    print(f"{dim:>6d}  {abs(rms_ref - rms_int).max():>8.4f}  {abs(ln_ref - ln_int).max():>8.4f}")

   dim   RMS err    LN err
    64    0.0054    0.0059
   256    0.0061    0.0055
  1024    0.0058    0.0065
  4096    0.0065    0.0056


## C kernel

Both norms share `isqrt_fixed` (clz + LUT). OpenMP across rows.

| step | ops | count |
|:-----|:----|------:|
| sum of squares | int64 mul + acc | n |
| mean | int div | 1 |
| 1/sqrt via LUT | clz + index + shift | 1 |
| scale | int mul + shift | 2n |

LayerNorm adds: mean (n adds + 1 div), centering (n subs).